## Perform trajectory analysis on VASA data

In [ ]:
import sys
import random
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", UserWarning)

import random

random.seed(10)

from anndata import AnnData
import numpy as np
import pandas as pd
import scanpy as sc
import logging
#import decoupler as dc
import cellrank as cr
import scvelo as scv
import matplotlib.pyplot as plt
import seaborn
seaborn.reset_orig()
%matplotlib inline

logging.getLogger('matplotlib.font_manager').disabled = True

import warnings
import scanpy as sc
import matplotlib as mpl

# Silence warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

# ---- Scanpy global settings ----
sc.settings.verbosity = 2
sc.settings.autoshow = False
sc.settings.set_figure_params(
    dpi=150,
    dpi_save=300,
    format="pdf",
    facecolor="white",
    frameon=False,
    vector_friendly=True,
    fontsize=10,
    figsize=(4, 4),
    transparent=True,
)

# ---- Matplotlib global settings ----
mpl.rcParams.update(
    {
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.fontsize": 6,
        "axes.titlesize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
    }
)

sc.settings.figdir = "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures"

## Load data

In [ ]:
# Load the dataset
adata_orig = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.h5ad")

# Remove SMOC2+ cells
adata_orig = adata_orig[adata_orig.obs["cell_type_initial"] != "SMOC2+ cells"]

In [ ]:
# Load data
adata = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.cellrank.h5ad")
adata = adata.raw.to_adata()

# Add approriate colors for the terminal states
adata.uns['term_states_fwd_colors'] = ["#ee8865","#85a7cf", "#6ac077", "#58B6D7"]

# Standard preprocessing
adata.raw = adata.copy()
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

In [ ]:
# Add umap embedding from adata_orig back to adata
adata.obsm["X_umap"] = adata_orig[adata.obs_names].obsm["X_umap"]

In [ ]:
# Filter genes within less than 50 cells
sc.pp.filter_genes(adata, min_cells=50)

In [ ]:
adata.raw = adata.copy()
adata.layers["counts"] = adata.X.copy()

In [ ]:
# Normalize and log1p
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pl.umap(adata, color=["FGFR1", "KLB"], cmap="Reds", na_color="#cccccc", vmin=1e-9)

In [ ]:
def parse_metric(x):
    parts = x.split("_")
    return pd.Series({
        "cell_type": parts[0],
        "metric": "_".join(parts[1:]) if len(parts) > 2 else parts[1]
    })

delta_df = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.cellrank.lineage.drivers.csv").rename(columns={"Unnamed: 0": "Gene"})
delta_df = delta_df.melt(id_vars=["Gene"], var_name="metric", value_name="value")
delta_df[["cell_type", "metric"]] = delta_df["metric"].apply(lambda x: parse_metric(x))
delta_df = pd.pivot_table(delta_df, index=["Gene", "cell_type"], columns=["metric"], values="value").reset_index()
delta_df["abs_corr"] = delta_df["corr"].abs()

In [ ]:
delta_df.query("qval < 0.05").groupby("cell_type").apply(lambda x: x.sort_values("abs_corr", ascending=False).head(10))

In [ ]:
from cellrank._utils import Lineage

lineages = Lineage.from_adata(adata)
fate_prob_df = pd.DataFrame(lineages, index=adata.obs_names, columns=lineages.names)
adata.obs[fate_prob_df.columns] = fate_prob_df

In [ ]:
adata.obs[fate_prob_df.columns]

In [ ]:
import scipy

# load lineages
lineages = Lineage.from_adata(adata)

# hard assignment: argmax fate
adata.obs["hard_fate"] = lineages.X.argmax(1).astype(str)

# add progenitor state: e.g. where all probabilities are still diffuse
entropy = -(lineages.X * np.log(lineages.X + 1e-12)).sum(1)  # per-cell entropy
branch_threshold = np.quantile(entropy, 0.8)  # tweak this threshold
adata.obs["hard_fate"] = np.where(
    entropy > branch_threshold, "progenitor", adata.obs["hard_fate"]
)

In [ ]:
# Define the colormap
ct_colors = {
    "Early EECs": "#d3d19a",
    "K cells": "#58B6D7",
    "I cells": "#629BD2",
    "I/K cells": "#73cdd1",
    "Late I/K cells": "#73cdd1",
    "Early I/K cells": "#c3e4dd",
    "Early X cells": "#c6c6e0",
    "X cells": "#85a7cf",
    "Late X cells": "#85a7cf",
    "Early ECs": "#f8bbaa",
    "ECs": "#ee8865",
    "Late ECs": "#ee8865",
    "D cells": "#6ac077"
}

In [ ]:
sc.pl.umap(adata, color="cell_type_initial", palette=ct_colors, save="_cell_types_scanpy_v3.pdf")

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

colors = ["#374d37", "#51a144", "#fae503", "#d53827", "#b23e22"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

sc.pl.umap(adata, color="DT", cmap=cmap, save="_DT_v3.pdf", size=30)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#dbe9f6",
    "#9ecae1",
    "#6baed6",
    "#3182bd",
    "#08519c",
]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

sc.pl.umap(adata, color="DT", cmap=cmap, save="_dpt_pseudotime_v3.pdf", size=30)

In [ ]:
sc.pl.umap(adata, color="dpt_pseudotime", palette=ct_colors, cmap="magma", save="_dpt_pseudotime_v3.pdf")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

lineage_cols = {
    "K cells": "#58B6D7",
    "Late X cells": "#85a7cf",
    "Late ECs": "#ee8865",
    "D cells": "#6ac077"
}

coords = adata.obsm["X_umap"]

fig, ax = plt.subplots(figsize=(4, 4))

# plot lineages one on top of another
for lin, col in lineage_cols.items():
    ax.scatter(
        coords[:,0], coords[:,1],
        c=col,
        s=10,
        alpha=adata.obs[lin].values,  # fade by fate probability
        label=lin
    )

ax.set_title("Fate probabilities (merged)")
ax.axis("off")
plt.savefig(Path(sc.settings.figdir) / "vasa_fate_probabilities_merged_v3.pdf", bbox_inches="tight")

In [ ]:
sc.pl.umap(adata, color="term_states_fwd_probs", cmap="viridis")#, save="_term_states_fwd_probs.pdf")

## Plot correlation of pseudotime and real time

In [ ]:
import scipy

fig, ax = plt.subplots(figsize=(6, 6))


# scatter plot from scanpy
sc.pl.scatter(
    adata,
    x="dpt_pseudotime",
    y="DT",
    color="cell_type_initial",
    ax=ax,
    show=False  # prevent Scanpy from opening a new figure
)

# get data for regression
x = adata.obs["dpt_pseudotime"].to_numpy()
y = adata.obs["DT"].to_numpy()

# compute correlation
corr, pval = scipy.stats.pearsonr(x, y)
print(f"Pearson correlation: {corr:.3f} (p-value: {pval:.3e})")

# plot y=x line
lims = [0, 1]
ax.plot(lims, lims, '--', color='black', zorder=0)

# labels, legend, grid
ax.set_xlabel("Pseudotime")
ax.set_ylabel("Realtime")
ax.grid(False)

plt.savefig("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/realtime_vs_pseudotime_v3.pdf", dpi=300)


## Plot distribution of DT by cell type

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
# Order cell types by pseudotime
cell_type_order = (
    adata.obs.groupby("cell_type_initial")["DT"].median().sort_values().index.tolist()
)
sc.pl.violin(
    adata,
    keys="DT",
    groupby="cell_type_initial",
    order=cell_type_order,
    rotation=90,
    ax=ax,
    show=False  
)

ax.set_ylabel("Realtime")
plt.tight_layout()
plt.grid(False)
plt.savefig("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/realtime_per_celltype_v3.pdf")

## Plot gene expression along pseudotime

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import marsilea as ma
import marsilea.plotter as mp
from matplotlib import cm
from cellrank._utils import Lineage
from pygam import LinearGAM, s


def fit_gam(x, y, lam=0.6, n_splines=10):
    """
    Fit a GAM to expression data y over pseudotime x.
    
    Parameters
    ----------
    x : array-like
        Pseudotime values (1D).
    y : array-like
        Expression values (1D).
    lam : float
        Smoothing parameter (higher = smoother).
    n_splines : int
        Number of spline basis functions.
        
    Returns
    -------
    y_pred : np.ndarray
        Smoothed expression values predicted over x.
    """
    # Reshape inputs
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y).ravel()

    # Fit GAM with Gamma distribution and log link
    gam = LinearGAM(s(0, n_splines=n_splines), lam=lam)
    gam.fit(x, y)

    return gam.predict(x)


def plot_exp_along_pseudotime(
    adata,
    genes,
    lineage,
    terminal_state=None,
    time_key="dpt_pseudotime",
    n_bins=20,
    cmap="RdBu_r",
    show_genes=True,
    min_weight=0.0,
    annotations=None,
    zscale=False,
    gam_lam=0.6,
    gam_splines=10,
):
    if annotations is None:
        annotations = {}

    # pseudotime & lineage weights
    pt_full = adata.obs[time_key].to_numpy()
    w_full = Lineage.from_adata(adata)[lineage].X.squeeze().astype(float)

    # clip pseudotime
    if terminal_state is None:
        terminal_state = lineage
    max_pt = adata.obs.loc[adata.obs["cell_type_initial"] == terminal_state, time_key].max()

    keep = (pt_full <= max_pt) & (w_full >= min_weight)
    pt = pt_full[keep]
    weights = w_full[keep]

    # helper to gene vector
    def _get_gene_vec(g):
        x = adata[:, g].X
        x = x.toarray().ravel() if hasattr(x, "toarray") else np.asarray(x).ravel()
        return x[keep]

    # binning
    n_bins = int(n_bins)
    bins = np.linspace(0.0, max_pt, n_bins + 1)
    bin_idx = np.clip(np.digitize(pt, bins, right=False) - 1, 0, n_bins - 1)

    # weighted mean expression per gene × bin
    G = list(genes)
    M = np.full((len(G), n_bins), np.nan)
    for i, g in enumerate(G):
        x = _get_gene_vec(g)
        for b in range(n_bins):
            m = bin_idx == b
            if m.any():
                ww, xx = weights[m], x[m]
                M[i, b] = np.average(xx, weights=ww) if ww.sum() > 0 else np.nan

    # fill gaps and scale
    M = pd.DataFrame(M).T.fillna(method="ffill").fillna(method="bfill").T.to_numpy()

    # midpoints of bins for labeling
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    # smooth with GAM for each row
    smoothed = []
    for row in M:
        y_pred = fit_gam(bin_centers, row, lam=gam_lam, n_splines=gam_splines)
        smoothed.append(y_pred)
    M = np.vstack(smoothed)

    if zscale:
        mu, sd = M.mean(axis=1, keepdims=True), M.std(axis=1, keepdims=True)
        sd[sd == 0] = 1.0
        M = (M - mu) / sd
    else:
        mn, mx = M.min(axis=1, keepdims=True), M.max(axis=1, keepdims=True)
        M = (M - mn) / (mx - mn + 1e-8)
    
    # order genes
    order = np.nanargmax(M, axis=1).argsort()[::-1]
    M = M[order]
    G_ordered = [G[i] for i in order]
    
    # build expression DataFrame for marsilea (rows=genes, cols=bins)
    expr_df = pd.DataFrame(M, index=G_ordered, columns=[f"{c:.2f}" for c in bin_centers])

    # annotation tracks
    anno_dfs = {}
    for k, arr in annotations.items():
        arr_full = adata.obs[k].to_numpy()[keep]
        bar = np.full(n_bins, np.nan)
        for b in range(n_bins):
            m = bin_idx == b
            if m.any():
                ww, xx = weights[m], arr_full[m]
                bar[b] = np.average(xx, weights=ww) if ww.sum() > 0 else np.nan
        bar = pd.Series(bar).fillna(method="ffill").fillna(method="bfill").to_numpy()
        lo, hi = np.nanmin(bar), np.nanmax(bar)
        anno_dfs[k] = np.clip(bar, 0, 1)

    # build marsilea heatmap
    if zscale:
        h = ma.Heatmap(expr_df, cmap=cmap, vmin=-2, center=0, vmax=2,
                       label="Normalized expression", height=10, width=5, 
                       rasterized=True)
    else:
        h = ma.Heatmap(expr_df, cmap=cmap, vmin=0, center=0.5, vmax=1,
                       label="Normalized expression", height=10, width=5,
                       rasterized=True)
    # add top annotation bars
    for k, v in anno_dfs.items():
        cmap_k = annotations[k]
        cmap_k = cm.get_cmap(cmap_k) if isinstance(cmap_k, str) else cmap_k

        h.add_top(
            mp.ColorMesh(v[np.newaxis, :].ravel(), cmap=cmap_k, label=k, vmin=0, vmax=1),
            size=0.2, pad=0.01, legend=True
        )

    # add labels
    if show_genes:
        h.add_right(mp.Labels(expr_df.index, align="center"), pad=0.1)

    h.add_legends()
    h.render()
    return h

In [ ]:
annotations = {
    "dpt_pseudotime": "Blues",
    "DT": "GnBu"
}

In [ ]:
heatmaps = []
for cell_type_query in ["Late ECs", "Late X cells", "D cells",  "K cells"]:
    genes = delta_df.query(f"qval < 0.05 & cell_type == '{cell_type_query}' & abs_corr > 0.1")["Gene"].unique()
    genes = [g for g in genes if g in adata.var_names]
    print(f"{cell_type_query}: {len(genes)} genes")
    if len(genes) > 0:
        heatmaps.append(plot_exp_along_pseudotime(adata, genes, time_key="dpt_pseudotime", 
                                    lineage=cell_type_query, n_bins=50, cmap="RdBu_r", show_genes=False, zscale=True,
                                    annotations=annotations))

In [ ]:
(heatmaps[0] + .5 + heatmaps[1] + .5 + heatmaps[2] + .5 + heatmaps[3]).render()

plt.savefig("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/vasa_cellrank_driver_genes_heatmap_v3.pdf", bbox_inches='tight', dpi=300)